# Blinde POS-Arbeitsdateien erzeugen

**Ordnerkontext:** liegt in `00_werkzeuge/`, liest aus
`02_quelldaten/`, schreibt nach `03_annotation_laufend/pos/tranche<N>/`.

## Konfiguration


In [ ]:
from pathlib import Path

BASIS_QUELL = Path("..") / "02_quelldaten"
CONLL       = BASIS_QUELL / "goldstandard_v1.conll"

TRANCHE     = 1                        
BEARBEITER  = ["A4", "A5", "A6"]       
TAGSET      = "upos_adj"               
ROTIEREN    = True                     
UEBERSCHREIBEN = False                 

BASIS_ARBEIT = (Path("..") / "03_annotation_laufend" / "pos"
                / f"tranche{TRANCHE}")

assert CONLL.exists(), f"{CONLL} nicht gefunden — aus 00_werkzeuge/ starten?"
print(f"Quelle:  {CONLL}")
print(f"Ziel:    {BASIS_ARBEIT}")
print(f"Tranche: {TRANCHE} | Tagset: {TAGSET} | Bearbeitende: {BEARBEITER}")
print("Ueberschreiben:", "JA" if UEBERSCHREIBEN else "nein (Schutz aktiv)")

Quelle:  ../02_quelldaten/goldstandard_v1.conll
Ziel:    ../03_annotation_laufend/pos/tranche1
Tranche: 1 | Tagset: upos_adj | Bearbeitende: ['A4', 'A5', 'A6']
Ueberschreiben: nein (Schutz aktiv)


## Schritt 1 — Goldstandard einlesen und POS-Spalte verwerfen


In [ ]:
zeilen = CONLL.read_text(encoding="utf-8").splitlines()

bloecke = []          # je Block: Metadaten und Saetze
meta, saetze, akt = {}, [], []
lauf = {"token": 0, "satz": 0, "block": 0}   # stabile Zaehler

def satz_abschliessen():
    global akt
    if akt:
        lauf["satz"] += 1
        text = " ".join(tk["token"] for tk in akt)
        for nr, tok in enumerate(akt):
            tok["satz"] = lauf["satz"]
            tok["token_nr"] = nr
            tok["satz_text"] = text if nr == 0 else ""
        saetze.append(akt)
        akt = []

def block_abschliessen():
    global meta, saetze
    satz_abschliessen()
    if saetze:
        lauf["block"] += 1
        for s in saetze:
            for tok in s:
                tok["block_nr"] = lauf["block"]
        bloecke.append({"meta": dict(meta), "saetze": saetze})
    saetze = []

for z in zeilen:
    if z.startswith("#"):
        if " = " in z:
            schluessel, wert = z[1:].split(" = ", 1)
            schluessel = schluessel.strip()
            if schluessel == "erzaehlung":      # neuer Block beginnt
                block_abschliessen()
            meta[schluessel] = wert.strip()
        continue
    if not z.strip():
        satz_abschliessen()
        continue
    akt.append({"id": lauf["token"], "token": z.split("\t")[0]})
    lauf["token"] += 1
block_abschliessen()

n_saetze = sum(len(b["saetze"]) for b in bloecke)
n_token  = sum(len(s) for b in bloecke for s in b["saetze"])
print(f"Gelesen: {len(bloecke)} Bloecke, {n_saetze} Saetze, {n_token} Token")

beispiel = [tk["token"] for tk in bloecke[0]["saetze"][0][:6]]
print(f"Erste Token: {beispiel}")
print("Uebernommene Spalten je Zeile: 1 (Token) — keine Tagwerte gelesen.")

tranche_bloecke = [b for b in bloecke if b["meta"].get("tranche") == str(TRANCHE)]
t_saetze = sum(len(b["saetze"]) for b in tranche_bloecke)
t_token  = sum(len(s) for b in tranche_bloecke for s in b["saetze"])
print(f"\nTranche {TRANCHE}: {len(tranche_bloecke)} Bloecke, "
      f"{t_saetze} Saetze, {t_token} Token")
assert tranche_bloecke, f"Keine Bloecke mit tranche = {TRANCHE} gefunden."

Gelesen: 40 Bloecke, 150 Saetze, 4137 Token
Erste Token: ['Plötzlich', 'aber', 'ließ', 'sich', 'in', 'dem']
Uebernommene Spalten je Zeile: 1 (Token) — keine Tagwerte gelesen.

Tranche 1: 20 Bloecke, 75 Saetze, 1794 Token


## Schritt 2 — Tagliste bereitstellen


In [4]:
UPOS_REST = ["ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", "NUM",
             "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X"]

STTS = ["ADJA", "ADJD", "ADV", "APPR", "APPRART", "APPO", "APZR", "ART",
        "CARD", "FM", "ITJ", "KOUI", "KOUS", "KON", "KOKOM", "NN", "NE",
        "PDS", "PDAT", "PIS", "PIAT", "PPER", "PPOSAT", "PPOSS", "PRELS",
        "PRELAT", "PRF", "PWS", "PWAT", "PWAV", "PAV", "PTKZU", "PTKNEG",
        "PTKVZ", "PTKANT", "PTKA", "TRUNC", "VVFIN", "VVIMP", "VVINF",
        "VVIZU", "VVPP", "VAFIN", "VAIMP", "VAINF", "VAPP", "VMFIN",
        "VMINF", "VMPP", "XY", "$,", "$.", "$("]

if TAGSET == "upos_adj":
    TAGS = sorted(["ADJA", "ADJD"] + UPOS_REST)
elif TAGSET == "upos":
    TAGS = sorted(["ADJ"] + UPOS_REST)
elif TAGSET == "stts":
    TAGS = STTS
else:
    raise ValueError(f"Unbekanntes Tagset: {TAGSET}")

print(f"Tagset '{TAGSET}': {len(TAGS)} Tags")
print("  " + "  ".join(TAGS))

Tagset 'upos_adj': 18 Tags
  ADJA  ADJD  ADP  ADV  AUX  CCONJ  DET  INTJ  NOUN  NUM  PART  PRON  PROPN  PUNCT  SCONJ  SYM  VERB  X


## Schritt 3 — Blockreihenfolge je Person festlegen


In [5]:
reihenfolgen = {}
for i, person in enumerate(BEARBEITER):
    if ROTIEREN:
        versatz = round(i * len(tranche_bloecke) / len(BEARBEITER))
    else:
        versatz = 0
    reihenfolgen[person] = tranche_bloecke[versatz:] + tranche_bloecke[:versatz]

for person, folge in reihenfolgen.items():
    #Blockkennungen der Vorlage sind nicht eindeutig, daher laufende Nummer
    nummern = [b["saetze"][0][0]["block_nr"] for b in folge]
    print(f"  {person}: beginnt mit Block {nummern[0]:>2} | Folge "
          + " ".join(str(n) for n in nummern[:10]) + " ...")

  A4: beginnt mit Block  2 | Folge 2 3 6 7 9 11 14 15 17 20 ...
  A5: beginnt mit Block 15 | Folge 15 17 20 21 24 25 27 30 31 33 ...
  A6: beginnt mit Block 27 | Folge 27 30 31 33 36 38 40 2 3 6 ...


## Schritt 4 — Arbeitsmappen bauen


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Protection
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

SPALTEN = ["id", "block", "satz", "token_nr", "token", "tag", "unsicher",
           "kommentar", "satz_text"]
BREITEN = {"id": 7, "block": 8, "satz": 7, "token_nr": 9, "token": 22,
           "tag": 12, "unsicher": 10, "kommentar": 28, "satz_text": 70}
EDITIERBAR = {"tag", "unsicher", "kommentar"}


def baue_mappe(person, folge, ziel):
    wb = Workbook()
    ws = wb.active
    ws.title = "POS"

    kopf_font = Font(name="Arial", bold=True, color="FFFFFF")
    kopf_fill = PatternFill("solid", fgColor="404040")
    edit_fill = PatternFill("solid", fgColor="FFF2CC")   # gelb = ausfüllen
    satz_fill = PatternFill("solid", fgColor="F2F2F2")   # graue Satzstreifen
    zell_font = Font(name="Arial")

    for j, name in enumerate(SPALTEN, start=1):
        c = ws.cell(row=1, column=j, value=name)
        c.font, c.fill = kopf_font, kopf_fill
        c.protection = Protection(locked=True)
        ws.column_dimensions[get_column_letter(j)].width = BREITEN[name]

    zeile = 2
    satz_nr = 0
    for block in folge:
        kennung = block["meta"].get("block", "?")
        for satz in block["saetze"]:
            satz_nr += 1
            streifen = (satz_nr % 2 == 0)
            for tk in satz:
                werte = {"id": tk["id"], "block": kennung,
                         "satz": tk["satz"], "token_nr": tk["token_nr"],
                         "token": tk["token"], "tag": "", "unsicher": "",
                         "kommentar": "", "satz_text": tk["satz_text"]}
                for j, name in enumerate(SPALTEN, start=1):
                    c = ws.cell(row=zeile, column=j, value=werte[name])
                    c.font = zell_font
                    if name == "satz_text":
                        c.alignment = Alignment(vertical="top")
                    if name in EDITIERBAR:
                        c.fill = edit_fill
                        c.protection = Protection(locked=False)
                    else:
                        c.protection = Protection(locked=True)
                        if streifen:
                            c.fill = satz_fill
                zeile += 1

    letzte = zeile - 1
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(SPALTEN))}{letzte}"

    #Auswahlliste fuer die Tagspalte; lange Listen wandern auf ein Hilfsblatt
    tag_sp = get_column_letter(SPALTEN.index("tag") + 1)
    inline = ",".join(TAGS)
    if len(inline) <= 240:
        dv = DataValidation(type="list", formula1=f'"{inline}"',
                            allow_blank=True, showDropDown=False)
    else:
        hilfs = wb.create_sheet("Tagliste")
        for i, tag in enumerate(TAGS, start=1):
            hilfs.cell(row=i, column=1, value=tag).font = zell_font
        hilfs.protection.sheet = True
        dv = DataValidation(
            type="list", formula1=f"Tagliste!$A$1:$A${len(TAGS)}",
            allow_blank=True, showDropDown=False)
    dv.showErrorMessage = False          # Freitext bleibt moeglich
    dv.promptTitle = "Wortart"
    dv.prompt = "Tag aus der Liste waehlen"
    ws.add_data_validation(dv)
    dv.add(f"{tag_sp}2:{tag_sp}{letzte}")

    #Blattschutz (invertierte Flag-Semantik, siehe Erlaeuterung oben)
    ws.protection.sheet = True
    ws.protection.selectUnlockedCells = False
    ws.protection.selectLockedCells = False
    ws.protection.formatColumns = False
    ws.protection.formatRows = False
    ws.protection.autoFilter = False
    ws.protection.sort = True
    ws.protection.deleteRows = True
    ws.protection.insertRows = True

    #Anleitungsblatt
    leg = wb.create_sheet("Anleitung", 0)
    text = [
        (f"POS-Annotation — {person}, Tranche {TRANCHE}", ""),
        ("", ""),
        ("Auszufuellen:", "nur die GELB hinterlegten Spalten"),
        ("  tag", "Wortart aus der Auswahlliste"),
        ("  unsicher", "x setzen, wenn die Entscheidung unklar bleibt"),
        ("  kommentar", "optional, kurze Begruendung"),
        ("", ""),
        ("Wichtig:", "Jede Zeile erhaelt genau ein Tag."),
        ("", "Die Tokenspalte NICHT veraendern."),
        ("", "Blattschutz NICHT aufheben, nicht sortieren."),
        ("", ""),
        ("Kernunterscheidung:", ""),
        ("  ADJA", "vor einem Substantiv, flektiert: das grosse Haus"),
        ("  ADJD", "praedikativ oder adverbial: er ist gross"),
        ("  NOUN", "auch bei Nominalisierung: die Alte sprach"),
        ("", ""),
        ("Unsicher?", "Tag nach bestem Ermessen setzen UND x in 'unsicher'."),
        ("", "Gekennzeichnete Unsicherheit schadet nicht, Raten schon."),
        ("", ""),
        ("Satzkontext:", "Spalte 'satz_text', jeweils in der ersten"),
        ("", "Zeile jedes Satzes."),
        ("Volle Regeln:", "goldstandard_v1_pos_richtlinie.md"),
        ("Reihenfolge:", "Die Bloecke stehen in persoenlicher Reihenfolge."),
    ]
    for i, (a, b) in enumerate(text, start=1):
        ca = leg.cell(row=i, column=1, value=a)
        cb = leg.cell(row=i, column=2, value=b)
        ca.font = Font(name="Arial", bold=(i == 1 or a.endswith(":")))
        cb.font = Font(name="Arial")
    leg.column_dimensions["A"].width = 22
    leg.column_dimensions["B"].width = 58
    leg.protection.sheet = True

    ziel.parent.mkdir(parents=True, exist_ok=True)
    wb.save(ziel)
    return letzte - 1


print("Baufunktion definiert.")

Baufunktion definiert.


## Schritt 5 — Mappen erzeugen


In [7]:
ergebnisse = {}
print("Arbeitsmappen:")
for person, folge in reihenfolgen.items():
    ziel = BASIS_ARBEIT / f"{person}_pos_tranche{TRANCHE}.xlsx"
    if ziel.exists() and not UEBERSCHREIBEN:
        print(f"  [uebersprungen] {ziel.name}  (existiert bereits)")
        continue
    anzahl = baue_mappe(person, folge, ziel)
    ergebnisse[person] = anzahl
    print(f"  [geschrieben  ] {ziel.name}  ({anzahl} Token)")

Arbeitsmappen:
  [geschrieben  ] A4_pos_tranche1.xlsx  (1794 Token)
  [geschrieben  ] A5_pos_tranche1.xlsx  (1794 Token)
  [geschrieben  ] A6_pos_tranche1.xlsx  (1794 Token)


## Schritt 6 — Kontrolle


In [8]:
from openpyxl import load_workbook

print("Kontrolle:")
for person in BEARBEITER:
    pfad = BASIS_ARBEIT / f"{person}_pos_tranche{TRANCHE}.xlsx"
    if not pfad.exists():
        print(f"  {person}: Datei fehlt")
        continue
    ws = load_workbook(pfad)["POS"]
    kopf = [c.value for c in ws[1]]
    sp = {name: kopf.index(name) + 1 for name in SPALTEN}
    zeilen_n = ws.max_row - 1

    gefuellt = sum(1 for i in range(2, ws.max_row + 1)
                   if ws.cell(row=i, column=sp["tag"]).value)
    schluessel = {ws.cell(row=i, column=sp["id"]).value
                  for i in range(2, ws.max_row + 1)}

    status = []
    status.append("Tokenzahl ok" if zeilen_n == t_token
                  else f"ABWEICHUNG {zeilen_n} statt {t_token}")
    status.append("Tagspalte leer" if gefuellt == 0
                  else f"WARNUNG: {gefuellt} Tags gesetzt")
    status.append("IDs eindeutig" if len(schluessel) == zeilen_n
                  else "WARNUNG: IDs nicht eindeutig")
    if "referenz_ids" not in dir():
        referenz_ids = schluessel
    status.append("IDs deckungsgleich" if schluessel == referenz_ids
                  else "WARNUNG: andere Tokenmenge als erste Mappe")
    print(f"  {person}: {zeilen_n} Token | " + " | ".join(status))

print()
print("Fuer Vergleich und Zusammenfuehrung wird nach der Spalte 'id' "
      "sortiert;")
print("die persoenliche Blockreihenfolge spielt dabei keine Rolle.")

Kontrolle:
  A4: 1794 Token | Tokenzahl ok | Tagspalte leer | IDs eindeutig | IDs deckungsgleich
  A5: 1794 Token | Tokenzahl ok | Tagspalte leer | IDs eindeutig | IDs deckungsgleich
  A6: 1794 Token | Tokenzahl ok | Tagspalte leer | IDs eindeutig | IDs deckungsgleich

Fuer Vergleich und Zusammenfuehrung wird nach der Spalte 'id' sortiert;
die persoenliche Blockreihenfolge spielt dabei keine Rolle.
